In [0]:
%sql
SELECT
    priority,
    referral_status,
    appointment_status,
    COUNT(*) AS patient_count,
    ROUND(AVG(waiting_days), 1) AS avg_waiting_days,
    MAX(waiting_days) AS max_waiting_days
FROM healthcare_demo.silver.patient_referral_details
GROUP BY priority, referral_status, appointment_status
ORDER BY priority;

priority,referral_status,appointment_status,patient_count,avg_waiting_days,max_waiting_days
Routine,Open,Attended,18,126.8,199
Routine,Open,Booked,14,105.0,179
Routine,Open,Cancelled,21,111.3,235
Urgent,Open,Cancelled,15,126.0,235
Urgent,Open,Booked,14,125.1,209
Urgent,Open,Attended,18,151.3,231


In [0]:
%sql
SELECT DISTINCT
    location_id,
    location_name,
    specialty,
    available_slots
FROM healthcare_demo.silver.patient_referral_details
ORDER BY location_id, specialty;

location_id,location_name,specialty,available_slots
L001,Central Clinic,Cardiology,5
L001,null,Dermatology,null
L001,null,Orthopaedics,null
L002,null,Cardiology,null
L002,null,Dermatology,null
L002,North Clinic,Orthopaedics,8
L003,null,Cardiology,null
L003,South Clinic,Dermatology,6
L003,null,Orthopaedics,null


In [0]:
%sql
SELECT
    patient_id,
    referral_id,
    priority,
    specialty,
    location_id,
    location_name,
    referral_date,
    waiting_days,
    appointment_status
FROM healthcare_demo.silver.patient_referral_details
WHERE appointment_id IS NULL
   OR LOWER(appointment_status) IN ('cancelled', 'waiting')
ORDER BY waiting_days DESC;

patient_id,referral_id,priority,specialty,location_id,location_name,referral_date,waiting_days,appointment_status
P0049,R0049,Urgent,Cardiology,L002,null,2026-05-18,235,Cancelled
P0078,R0078,Routine,Dermatology,L002,null,2026-05-15,235,Cancelled
P0062,R0062,Urgent,Dermatology,L002,null,2026-05-15,219,Cancelled
P0098,R0098,Urgent,Dermatology,L003,South Clinic,2026-07-09,170,Cancelled
P0045,R0045,Routine,Dermatology,L003,South Clinic,2026-06-08,165,Cancelled
P0082,R0082,Urgent,Dermatology,L001,null,2026-07-17,164,Cancelled
P0027,R0027,Routine,Orthopaedics,L002,North Clinic,2026-08-02,158,Cancelled
P0083,R0083,Urgent,Orthopaedics,L003,null,2026-07-08,155,Cancelled
P0052,R0052,Routine,Cardiology,L003,null,2026-07-22,148,Cancelled
P0071,R0071,Routine,Orthopaedics,L002,North Clinic,2026-08-03,147,Cancelled


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_table = "healthcare_demo.silver.patient_referral_details"

df = spark.table(silver_table)

# Extract confirmed hospital capacity
capacity_df = (
    df.select(
        "location_id",
        "specialty",
        "available_slots"
    )
    .filter(
        F.col("available_slots").isNotNull() &
        (F.col("available_slots") > 0)
    )
    .dropDuplicates(["location_id", "specialty"])
)

# Add hospital names using location IDs
capacity_df = capacity_df.withColumn(
    "location_name",
    F.when(F.col("location_id") == "L001", "Central Clinic")
     .when(F.col("location_id") == "L002", "North Clinic")
     .when(F.col("location_id") == "L003", "South Clinic")
)

display(capacity_df)

location_id,specialty,available_slots,location_name
L002,Orthopaedics,8,North Clinic
L001,Cardiology,5,Central Clinic
L003,Dermatology,6,South Clinic


In [0]:
waiting_df = (
    df.filter(
        (F.col("appointment_id").isNull()) |
        (F.lower(F.col("appointment_status")).isin(
            "cancelled", "waiting"
        ))
    )
    .select(
        "patient_id",
        "referral_id",
        "priority",
        "specialty",
        "location_id",
        "referral_date",
        "waiting_days"
    )
    .dropDuplicates(["referral_id"])
)

# Urgent patients first, then longest waiting time
priority_window = Window.partitionBy("specialty").orderBy(
    F.when(F.lower(F.col("priority")) == "urgent", 1)
     .otherwise(2),
    F.col("waiting_days").desc(),
    F.col("referral_date").asc()
)

waiting_df = waiting_df.withColumn(
    "priority_rank",
    F.row_number().over(priority_window)
)

display(waiting_df)

patient_id,referral_id,priority,specialty,location_id,referral_date,waiting_days,priority_rank
P0049,R0049,Urgent,Cardiology,L002,2026-05-18,235,1
P0055,R0055,Urgent,Cardiology,L001,2026-07-30,141,2
P0019,R0019,Urgent,Cardiology,L003,2026-08-30,106,3
P0087,R0087,Urgent,Cardiology,L003,2026-10-12,48,4
P0065,R0065,Urgent,Cardiology,L003,2026-10-26,32,5
P0052,R0052,Routine,Cardiology,L003,2026-07-22,148,6
P0054,R0054,Routine,Cardiology,L002,2026-08-28,123,7
P0038,R0038,Routine,Cardiology,L001,2026-08-26,117,8
P0088,R0088,Routine,Cardiology,L002,2026-09-08,86,9
P0058,R0058,Routine,Cardiology,L001,2026-09-08,86,10


In [0]:
# Prepare capacity data with separate column names
capacity_for_join = capacity_df.select(
    F.col("specialty").alias("capacity_specialty"),
    F.col("location_id").alias("recommended_location_id"),
    F.col("location_name").alias("recommended_location_name"),
    F.col("available_slots")
)

# Join waiting patients to matching specialty capacity
allocation_df = waiting_df.join(
    capacity_for_join,
    waiting_df["specialty"] == capacity_for_join["capacity_specialty"],
    "left"
)

# Remove duplicate specialty column
allocation_df = allocation_df.drop("capacity_specialty")

# Determine recommendation status
allocation_df = allocation_df.withColumn(
    "allocation_status",
    F.when(
        F.col("available_slots").isNull(),
        "Capacity information unavailable"
    ).when(
        F.col("priority_rank") <= F.col("available_slots"),
        "Recommended"
    ).otherwise(
        "Waiting for capacity"
    )
)

# Explain the recommendation
allocation_df = allocation_df.withColumn(
    "recommendation_reason",
    F.when(
        F.col("allocation_status") == "Recommended",
        F.concat(
            F.lit("Specialty match. Priority rank: "),
            F.col("priority_rank").cast("string")
        )
    ).otherwise(
        F.col("allocation_status")
    )
)

allocation_df = allocation_df.withColumn(
    "recommended_at",
    F.current_timestamp()
)

display(allocation_df)

patient_id,referral_id,priority,specialty,location_id,referral_date,waiting_days,priority_rank,recommended_location_id,recommended_location_name,available_slots,allocation_status,recommendation_reason,recommended_at
P0049,R0049,Urgent,Cardiology,L002,2026-05-18,235,1,L001,Central Clinic,5,Recommended,Specialty match. Priority rank: 1,2026-11-10T19:22:09.473Z
P0055,R0055,Urgent,Cardiology,L001,2026-07-30,141,2,L001,Central Clinic,5,Recommended,Specialty match. Priority rank: 2,2026-11-10T19:22:09.473Z
P0019,R0019,Urgent,Cardiology,L003,2026-08-30,106,3,L001,Central Clinic,5,Recommended,Specialty match. Priority rank: 3,2026-11-10T19:22:09.473Z
P0087,R0087,Urgent,Cardiology,L003,2026-10-12,48,4,L001,Central Clinic,5,Recommended,Specialty match. Priority rank: 4,2026-11-10T19:22:09.473Z
P0065,R0065,Urgent,Cardiology,L003,2026-10-26,32,5,L001,Central Clinic,5,Recommended,Specialty match. Priority rank: 5,2026-11-10T19:22:09.473Z
P0052,R0052,Routine,Cardiology,L003,2026-07-22,148,6,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:09.473Z
P0054,R0054,Routine,Cardiology,L002,2026-08-28,123,7,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:09.473Z
P0038,R0038,Routine,Cardiology,L001,2026-08-26,117,8,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:09.473Z
P0088,R0088,Routine,Cardiology,L002,2026-09-08,86,9,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:09.473Z
P0058,R0058,Routine,Cardiology,L001,2026-09-08,86,10,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:09.473Z


In [0]:
# Prepare capacity columns for joining
capacity_for_join = capacity_df.select(
    "specialty",
    F.col("location_id").alias("recommended_location_id"),
    F.col("location_name").alias("recommended_location_name"),
    "available_slots"
)

# Match patients to hospitals offering their specialty
allocation_df = waiting_df.join(
    capacity_for_join,
    on="specialty",
    how="left"
)

# Determine recommendation status
allocation_df = allocation_df.withColumn(
    "allocation_status",
    F.when(
        F.col("available_slots").isNull(),
        "Capacity information unavailable"
    ).when(
        F.col("priority_rank") <= F.col("available_slots"),
        "Recommended"
    ).otherwise(
        "Waiting for capacity"
    )
)

# Add recommendation explanation
allocation_df = allocation_df.withColumn(
    "recommendation_reason",
    F.when(
        F.col("allocation_status") == "Recommended",
        F.concat(
            F.lit("Specialty match and confirmed capacity. Priority rank: "),
            F.col("priority_rank").cast("string")
        )
    ).otherwise(F.col("allocation_status"))
)

allocation_df = allocation_df.withColumn(
    "recommended_at",
    F.current_timestamp()
)

display(allocation_df)

specialty,patient_id,referral_id,priority,location_id,referral_date,waiting_days,priority_rank,recommended_location_id,recommended_location_name,available_slots,allocation_status,recommendation_reason,recommended_at
Cardiology,P0049,R0049,Urgent,L002,2026-05-18,235,1,L001,Central Clinic,5,Recommended,Specialty match and confirmed capacity. Priority rank: 1,2026-11-10T19:22:36.845Z
Cardiology,P0055,R0055,Urgent,L001,2026-07-30,141,2,L001,Central Clinic,5,Recommended,Specialty match and confirmed capacity. Priority rank: 2,2026-11-10T19:22:36.845Z
Cardiology,P0019,R0019,Urgent,L003,2026-08-30,106,3,L001,Central Clinic,5,Recommended,Specialty match and confirmed capacity. Priority rank: 3,2026-11-10T19:22:36.845Z
Cardiology,P0087,R0087,Urgent,L003,2026-10-12,48,4,L001,Central Clinic,5,Recommended,Specialty match and confirmed capacity. Priority rank: 4,2026-11-10T19:22:36.845Z
Cardiology,P0065,R0065,Urgent,L003,2026-10-26,32,5,L001,Central Clinic,5,Recommended,Specialty match and confirmed capacity. Priority rank: 5,2026-11-10T19:22:36.845Z
Cardiology,P0052,R0052,Routine,L003,2026-07-22,148,6,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:36.845Z
Cardiology,P0054,R0054,Routine,L002,2026-08-28,123,7,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:36.845Z
Cardiology,P0038,R0038,Routine,L001,2026-08-26,117,8,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:36.845Z
Cardiology,P0088,R0088,Routine,L002,2026-09-08,86,9,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:36.845Z
Cardiology,P0058,R0058,Routine,L001,2026-09-08,86,10,L001,Central Clinic,5,Waiting for capacity,Waiting for capacity,2026-11-10T19:22:36.845Z


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_demo.gold")

(
    allocation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("healthcare_demo.gold.appointment_recommendations")
)

print("Patient allocation recommendations saved successfully")

Patient allocation recommendations saved successfully


In [0]:
%sql
SELECT
    specialty,
    recommended_location_name,
    allocation_status,
    COUNT(*) AS patient_count
FROM healthcare_demo.gold.appointment_recommendations
GROUP BY
    specialty,
    recommended_location_name,
    allocation_status
ORDER BY specialty;

specialty,recommended_location_name,allocation_status,patient_count
Cardiology,Central Clinic,Recommended,5
Cardiology,Central Clinic,Waiting for capacity,8
Dermatology,South Clinic,Waiting for capacity,7
Dermatology,South Clinic,Recommended,6
Orthopaedics,North Clinic,Recommended,8
Orthopaedics,North Clinic,Waiting for capacity,2


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Rank patients within each specialty
priority_window = (
    Window
    .partitionBy("specialty")
    .orderBy(
        F.when(F.col("priority") == "Urgent", 1).otherwise(2),
        F.col("waiting_days").desc(),
        F.col("referral_date").asc()
    )
)

waiting_ranked = waiting_df.withColumn(
    "priority_rank",
    F.row_number().over(priority_window)
)

# Keep only confirmed capacity records
capacity_for_join = capacity_df.select(
    "specialty",
    F.col("location_id").alias("recommended_location_id"),
    F.col("location_name").alias("recommended_location_name"),
    "available_slots"
)

# Match patients to their specialty clinic
allocation_df = waiting_ranked.join(
    capacity_for_join,
    on="specialty",
    how="left"
)

# Recommend patients within available capacity
allocation_df = (
    allocation_df
    .withColumn(
        "allocation_status",
        F.when(
            F.col("available_slots").isNull(),
            "Capacity information unavailable"
        ).when(
            F.col("priority_rank") <= F.col("available_slots"),
            "Recommended"
        ).otherwise("Waiting for capacity")
    )
    .withColumn(
        "recommendation_reason",
        F.concat(
            F.lit("Specialty match; priority: "),
            F.col("priority"),
            F.lit("; waiting days: "),
            F.col("waiting_days").cast("string")
        )
    )
    .withColumn("recommended_at", F.current_timestamp())
)

display(allocation_df)

specialty,patient_id,referral_id,priority,location_id,referral_date,waiting_days,priority_rank,recommended_location_id,recommended_location_name,available_slots,allocation_status,recommendation_reason,recommended_at
Cardiology,P0049,R0049,Urgent,L002,2026-05-18,235,1,L001,Central Clinic,5,Recommended,Specialty match; priority: Urgent; waiting days: 235,2026-11-10T19:31:53.011Z
Cardiology,P0055,R0055,Urgent,L001,2026-07-30,141,2,L001,Central Clinic,5,Recommended,Specialty match; priority: Urgent; waiting days: 141,2026-11-10T19:31:53.011Z
Cardiology,P0019,R0019,Urgent,L003,2026-08-30,106,3,L001,Central Clinic,5,Recommended,Specialty match; priority: Urgent; waiting days: 106,2026-11-10T19:31:53.011Z
Cardiology,P0087,R0087,Urgent,L003,2026-10-12,48,4,L001,Central Clinic,5,Recommended,Specialty match; priority: Urgent; waiting days: 48,2026-11-10T19:31:53.011Z
Cardiology,P0065,R0065,Urgent,L003,2026-10-26,32,5,L001,Central Clinic,5,Recommended,Specialty match; priority: Urgent; waiting days: 32,2026-11-10T19:31:53.011Z
Cardiology,P0052,R0052,Routine,L003,2026-07-22,148,6,L001,Central Clinic,5,Waiting for capacity,Specialty match; priority: Routine; waiting days: 148,2026-11-10T19:31:53.011Z
Cardiology,P0054,R0054,Routine,L002,2026-08-28,123,7,L001,Central Clinic,5,Waiting for capacity,Specialty match; priority: Routine; waiting days: 123,2026-11-10T19:31:53.011Z
Cardiology,P0038,R0038,Routine,L001,2026-08-26,117,8,L001,Central Clinic,5,Waiting for capacity,Specialty match; priority: Routine; waiting days: 117,2026-11-10T19:31:53.011Z
Cardiology,P0088,R0088,Routine,L002,2026-09-08,86,9,L001,Central Clinic,5,Waiting for capacity,Specialty match; priority: Routine; waiting days: 86,2026-11-10T19:31:53.011Z
Cardiology,P0058,R0058,Routine,L001,2026-09-08,86,10,L001,Central Clinic,5,Waiting for capacity,Specialty match; priority: Routine; waiting days: 86,2026-11-10T19:31:53.011Z


In [0]:
display(
    allocation_df
    .groupBy("specialty", "allocation_status")
    .count()
    .orderBy("specialty", "allocation_status")
)

specialty,allocation_status,count
Cardiology,Recommended,5
Cardiology,Waiting for capacity,8
Dermatology,Recommended,6
Dermatology,Waiting for capacity,7
Orthopaedics,Recommended,8
Orthopaedics,Waiting for capacity,2


In [0]:

display(
    allocation_df
    .filter(F.col("allocation_status") == "Recommended")
    .select(
        "patient_id",
        "referral_id",
        "priority",
        "specialty",
        "waiting_days",
        "recommended_location_id",
        "recommended_location_name",
        "allocation_status",
        "recommendation_reason"
    )
    .orderBy(
        "specialty",
        "priority_rank"
    )
)

patient_id,referral_id,priority,specialty,waiting_days,recommended_location_id,recommended_location_name,allocation_status,recommendation_reason
P0049,R0049,Urgent,Cardiology,235,L001,Central Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 235
P0055,R0055,Urgent,Cardiology,141,L001,Central Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 141
P0019,R0019,Urgent,Cardiology,106,L001,Central Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 106
P0087,R0087,Urgent,Cardiology,48,L001,Central Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 48
P0065,R0065,Urgent,Cardiology,32,L001,Central Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 32
P0062,R0062,Urgent,Dermatology,219,L003,South Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 219
P0098,R0098,Urgent,Dermatology,170,L003,South Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 170
P0082,R0082,Urgent,Dermatology,164,L003,South Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 164
P0077,R0077,Urgent,Dermatology,130,L003,South Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 130
P0095,R0095,Urgent,Dermatology,98,L003,South Clinic,Recommended,Specialty match; priority: Urgent; waiting days: 98


In [0]:
allocation_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_demo.gold.appointment_recommendations")

print("Patient recommendations saved successfully")

Patient recommendations saved successfully


In [0]:
saved_recommendations = spark.table(
    "healthcare_demo.gold.appointment_recommendations"
)

display(
    saved_recommendations
    .groupBy("specialty", "allocation_status")
    .count()
    .orderBy("specialty", "allocation_status")
)

specialty,allocation_status,count
Cardiology,Recommended,5
Cardiology,Waiting for capacity,8
Dermatology,Recommended,6
Dermatology,Waiting for capacity,7
Orthopaedics,Recommended,8
Orthopaedics,Waiting for capacity,2


In [0]:

from pyspark.sql import functions as F

healthcare_report = (
    allocation_df
    .groupBy("specialty")
    .agg(
        F.first("recommended_location_id", ignorenulls=True).alias("location_id"),
        F.first("recommended_location_name", ignorenulls=True).alias("location_name"),
        F.max("available_slots").alias("available_slots"),
        F.countDistinct("referral_id").alias("total_waiting_referrals"),
        F.round(F.avg("waiting_days"), 1).alias("average_waiting_days"),
        F.max("waiting_days").alias("maximum_waiting_days"),
        F.sum(
            F.when(F.col("allocation_status") == "Recommended", 1)
             .otherwise(0)
        ).alias("recommended_patients"),
        F.sum(
            F.when(F.col("allocation_status") == "Waiting for capacity", 1)
             .otherwise(0)
        ).alias("patients_waiting_for_capacity")
    )
    .orderBy("specialty")
)

display(healthcare_report)


specialty,location_id,location_name,available_slots,total_waiting_referrals,average_waiting_days,maximum_waiting_days,recommended_patients,patients_waiting_for_capacity
Cardiology,L001,Central Clinic,5,13,101.8,235,5,8
Dermatology,L003,South Clinic,6,13,135.0,235,6,7
Orthopaedics,L002,North Clinic,8,10,114.8,158,8,2


In [0]:
healthcare_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_demo.gold.healthcare_allocation_summary")

print("Final healthcare allocation report saved successfully")

Final healthcare allocation report saved successfully


In [0]:

from pyspark.sql import functions as F

# Read Gold recommendations
recommendations_df = spark.table(
    "healthcare_demo.gold.appointment_recommendations"
)

# Check duplicate referral recommendations
duplicate_allocations = (
    recommendations_df
    .groupBy("referral_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check recommendations against clinic capacity
capacity_validation = (
    recommendations_df
    .filter(F.col("allocation_status") == "Recommended")
    .groupBy(
        "specialty",
        "recommended_location_id"
    )
    .agg(
        F.countDistinct("referral_id").alias("recommended_patients"),
        F.max("available_slots").alias("available_slots")
    )
    .withColumn(
        "result",
        F.when(
            F.col("recommended_patients") <= F.col("available_slots"),
            "PASS"
        ).otherwise("FAIL")
    )
)

# Check missing clinic recommendations
missing_clinic = recommendations_df.filter(
    (F.col("allocation_status") == "Recommended") &
    (
        F.col("recommended_location_id").isNull() |
        F.col("recommended_location_name").isNull()
    )
).count()

# Display capacity validation
display(capacity_validation)

# Display overall allocation checks
allocation_results = [
    ("Duplicate referral allocations", duplicate_allocations),
    ("Recommended patients missing clinic", missing_clinic),
    (
        "Capacity violations",
        capacity_validation.filter(F.col("result") == "FAIL").count()
    )
]

allocation_quality_df = spark.createDataFrame(
    allocation_results,
    ["quality_check", "failed_records"]
).withColumn(
    "result",
    F.when(F.col("failed_records") == 0, "PASS")
     .otherwise("FAIL")
)

display(allocation_quality_df)

if allocation_quality_df.filter(F.col("result") == "FAIL").count() == 0:
    print("Gold allocation validation PASSED")
else:
    print("Gold allocation validation FAILED")


specialty,recommended_location_id,recommended_patients,available_slots,result
Cardiology,L001,5,5,PASS
Dermatology,L003,6,6,PASS
Orthopaedics,L002,8,8,PASS


quality_check,failed_records,result
Duplicate referral allocations,0,PASS
Recommended patients missing clinic,0,PASS
Capacity violations,0,PASS


Gold allocation validation PASSED
